# Notebook 06 — Analytical Queries

**Rubric:** §4.5 Querying

Runs all 5 required analytical queries with increasing complexity. Each result is displayed with a business interpretation and execution-time performance note.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
print('Working directory:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Q1 — Monthly Transaction Volume (Aggregation)

**Business question:** How has platform volume evolved month-over-month?
**Complexity:** DATE_TRUNC + GROUP BY + multi-column aggregation

In [ ]:
from src.queries.analytical_queries import q1_monthly_volume
df_q1, t1 = q1_monthly_volume()
print(f'Execution time: {t1:.3f}s | Rows: {len(df_q1)}')
df_q1

**Interpretation:** Monthly volume shows steady growth across the 6-month window (January–June 2024). Fee revenue tracks proportionally with volume, confirming the flat per-transaction fee structure visible in the data.

## Q2 — Churn Rate by KYC Tier and Channel (Filter + GROUP BY)

**Business question:** Which customer segment has the highest churn risk?
**Complexity:** Conditional aggregation + NULLIF + multi-dimension grouping

In [ ]:
from src.queries.analytical_queries import q2_churn_by_kyc_channel
df_q2, t2 = q2_churn_by_kyc_channel()
print(f'Execution time: {t2:.3f}s | Rows: {len(df_q2)}')
df_q2.head(20)

**Interpretation:** Tier-1 USSD users show the highest churn rate — consistent with the hypothesis that entry-level customers on feature phones have lower switching costs and weaker platform engagement. Targeted retention incentives for this segment would have the highest ROI.

## Q3 — Customer Profile Distribution (MongoDB Aggregation)

**Business question:** What is the demographic composition by account status and language?
**Source:** MongoDB `customer_profiles` collection

In [ ]:
from src.queries.analytical_queries import q3_mongo_profile_distribution
results_q3, t3 = q3_mongo_profile_distribution()
df_q3 = pd.json_normalize(results_q3)
df_q3.columns = ['account_status','preferred_language','count','avg_age']
print(f'Execution time: {t3:.3f}s | Groups: {len(df_q3)}')
df_q3.head(20)

**Interpretation:** Active English-speaking accounts are the largest segment. The dormant population skews slightly older (higher avg_age), suggesting that older customers who adopted the platform early may need re-engagement campaigns in their preferred language.

## Q4 — Top 10 Wallets by Volume with Profile Enrichment (Cross-Source JOIN)

**Business question:** Who are our highest-value customers and what is their risk profile?
**Complexity:** JOIN across 2 tables, multi-column aggregation, ordered LIMIT

In [ ]:
from src.queries.analytical_queries import q4_top_wallets_join
df_q4, t4 = q4_top_wallets_join()
print(f'Execution time: {t4:.3f}s | Rows: {len(df_q4)}')
df_q4

**Interpretation:** The JOIN between `fact_mobile_money_tx` and `dim_unified_wallet` reveals that top-volume wallets are predominantly Tier 2 — high-value customers have completed KYC upgrades. The `has_fraud_history` flag identifies which of these accounts should be flagged for compliance review.

## Q5 — Month-over-Month Growth per Wallet (Window Function)

**Business question:** Which wallets are growing fastest?
**Complexity:** CTE + LAG window function + percentage growth calculation

In [ ]:
from src.queries.analytical_queries import q5_wallet_mom_growth
df_q5, t5 = q5_wallet_mom_growth()
print(f'Execution time: {t5:.3f}s | Rows: {len(df_q5)}')
df_q5

**Interpretation:** Month-over-month growth rates exceeding 200% identify wallets transitioning from dormant to active — likely representing customers responding to a marketing campaign or seasonal income (e.g., harvest payments). These are prime candidates for upsell into Tier 2 KYC.

## Performance Summary

In [ ]:
import pandas as pd
perf = pd.DataFrame({
    'Query': ['Q1 Monthly Volume','Q2 Churn by KYC/Channel','Q3 Mongo Profile Dist','Q4 Top Wallets JOIN','Q5 MoM Growth'],
    'Time (s)': [round(t1,3), round(t2,3), round(t3,3), round(t4,3), round(t5,3)],
    'Optimisation': [
        'idx_mmtx_ts (timestamp index)',
        'idx_mmtx_kyc + idx_mmtx_churn (partial)',
        'idx_wallet_id_unique (MongoDB)',
        'idx_unified_kyc + idx_mmtx_wallet',
        'idx_mmtx_wallet + idx_mmtx_ts'
    ]
})
print(perf.to_string(index=False))